In [45]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
import numpy as np

In [46]:
def scrape_autoscout(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all car listings on the page
    cars = soup.find_all('article', class_='cldt-summary-full-item')

    # Prepare a list to store the car data
    car_data = []

    for car in cars:
        # Extract the data from each car listing
        url = car.find('a', class_="ListItem_title__ndA4s")['href']
        url = 'https://www.autoscout24.es' + url
        location = car.find('span', class_="SellerInfo_address__leRMu")
        if location is None:
            location = car.find('span', class_="SellerInfo_private__THzvQ").text
        else:
            location = location.text

        title_element = car.find('h2')
        description = title_element.find('span', class_='ListItem_version__5EWfi').text.strip()
        title = title_element.text.replace(description, '').strip()
        try:
            brand = title.split()[0]
        except IndexError:
            brand = np.nan

        price = car.find('p', class_='Price_price__APlgs').text.strip()
        price = int(re.sub(r'\D', '', price))

        details_elements = car.find_all('span', class_='VehicleDetailTable_item__4n35N')
        details = [detail.text.strip() for detail in details_elements]
        mileage = details[0]
        try:
            mileage = int(re.sub(r'\D', '', mileage))
        except ValueError:
            mileage = np.nan
        transmission = details[1]
        year = details[2]
        # year = datetime.strptime(year, '%m/%Y')
        engine_type = details[3]
        try:
            cv = details[4]
            cv = int(re.search(r'(\d+) CV', cv).group(1))
        except AttributeError:
            cv = np.nan

        # Add the data to our list
        car_data.append([brand, title, description, price, mileage, transmission, year, engine_type, cv, location, url])

    # Convert the list to a DataFrame and return it
    return pd.DataFrame(car_data, columns=['brand', 'title', 'description', 'full_price', 'mileage', 'transmission', 'year', 'fuel', 'cv', 'location', 'url']
)

def scrape_multiple_pages(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find max page 
    page = soup.find('li', class_='pagination-item--disabled pagination-item--page-indicator')
               
    if page is None:
        return
    else:
        page= int(page.text.split()[-1])

    # Prepare a list to store the DataFrames
    dfs = []

    for i in range(1, page + 1):
        # Construct the URL for the current page
        url = base_url + '&page=' + str(i)

        # Scrape the current page and store the DataFrame
        df = scrape_autoscout(url)
        dfs.append(df)

    # Concatenate all the DataFrames
    master_df = pd.concat(dfs, ignore_index=True)

    return master_df


In [47]:
df = pd.DataFrame()
for i in range(50):
    url = f'https://www.autoscout24.es/lst?damaged_listing=exclude&desc=0&lat=40.41669&lon=-3.70035&powertype=kw&pricefrom={(i+20)*3}00&priceto={(i+21)*3}00&sort=price&zip=madrid&zipr=20'
    df = pd.concat([df, scrape_multiple_pages(url)],ignore_index=True)
    df = df.drop_duplicates()
    print(df.shape)
    
df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)

(46, 11)
(123, 11)
(198, 11)
(311, 11)
(454, 11)
(526, 11)
(699, 11)
(810, 11)
(966, 11)
(1272, 11)
(1395, 11)
(1705, 11)
(1951, 11)
(2255, 11)
(2612, 11)
(2788, 11)
(3167, 11)
(3398, 11)
(3730, 11)
(4118, 11)
(4482, 11)
(4872, 11)
(5272, 11)
(5672, 11)
(6072, 11)
(6446, 11)
(6846, 11)
(7169, 11)
(7559, 11)
(7959, 11)
(8297, 11)
(8687, 11)
(9087, 11)
(9487, 11)
(9887, 11)
(10287, 11)
(10687, 11)
(11087, 11)
(11487, 11)
(11887, 11)
(12255, 11)
(12639, 11)
(13039, 11)
(13336, 11)
(13717, 11)
(14035, 11)
(14435, 11)
(14636, 11)
(15015, 11)
(15394, 11)


In [48]:
# df = pd.DataFrame()
# for i in range(70):
#     url = f'https://www.autoscout24.es/lst?atype=C&body=4%2C5%2C12&cy=E&damaged_listing=exclude&desc=0&fregfrom=2017&kmfrom=20000&kmto=100000&lat=40.41669&lon=-3.70035&powerfrom=59&powertype=hp&pricefrom={(i+30)*2}00&priceto={(i+31)*2}00&search_id=49cs04bi9u&sort=price&source=homepage_search-mask&ustate=N%2CU&zip=madrid-%28comunidad-de-madrid%29&zipr=50'
#     df = pd.concat([df, scrape_multiple_pages(url)],ignore_index=True)
#     df = df.drop_duplicates()
#     print(df.shape)
    
# df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)